# 04 — Testing, numerical tolerances and benchmarking

**Goal:** understand how to test floating-point software and design fair performance experiments. This notebook does not publish fixed performance claims; results depend on your machine and numerical libraries.

In [ ]:
import platform
import time
import numpy as np
import matplotlib.pyplot as plt
from matrixlib import Matrix

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
np.show_config()

## 1. Why floating-point tests use tolerances

Floating-point operations round intermediate values. Algebraic identities should usually be checked with a tolerance rather than exact equality.

In [ ]:
A = np.array([[0.1, 0.2], [0.3, 0.4]])
left = (A + A) - A
print("exact elementwise equality:", np.array_equal(left, A))
print("tolerance-based comparison:", np.allclose(left, A))
np.testing.assert_allclose(left, A, rtol=1e-12, atol=1e-12)

## 2. Test algebraic invariants

An invariant is a property that should hold for many valid inputs. These are useful in unit and property-based tests.

In [ ]:
rng = np.random.default_rng(2026)
for size in [2, 3, 5]:
    A = rng.normal(size=(size, size))
    B = rng.normal(size=(size, size))
    # Transpose of a product reverses the factor order.
    np.testing.assert_allclose((A @ B).T, B.T @ A.T, rtol=1e-10, atol=1e-10)
    # Inverse identity for a well-conditioned random example.
    shifted = A + size * np.eye(size)
    np.testing.assert_allclose(shifted @ np.linalg.inv(shifted), np.eye(size),
                               rtol=1e-9, atol=1e-9)
print("All demonstration invariants passed.")

## 3. A simple reproducible timing helper

A useful benchmark records workload size, repetitions, environment, and timing method. Run multiple repetitions and compare medians. Do not compare unlike workloads.

In [ ]:
def median_runtime(function, repeats=7):
    """Return the median wall-clock time in seconds after one warm-up."""
    function()  # warm-up
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function()
        samples.append(time.perf_counter() - start)
    return float(np.median(samples)), np.asarray(samples)

rng = np.random.default_rng(123)
sizes = [16, 32, 64, 128]
results = []
for size in sizes:
    A = rng.normal(size=(size, size))
    B = rng.normal(size=(size, size))
    wrapped_A, wrapped_B = Matrix(A), Matrix(B)
    direct_time, _ = median_runtime(lambda: A @ B)
    wrapper_time, _ = median_runtime(lambda: wrapped_A @ wrapped_B)
    results.append((size, direct_time, wrapper_time))
results

## 4. Plot measured results

This plot displays the measurements from the current run only. Small timings are noisy; repeat the experiment and avoid generalizing from a single machine.

In [ ]:
sizes = [row[0] for row in results]
direct = [row[1] for row in results]
wrapped = [row[2] for row in results]

plt.figure(figsize=(6, 4))
plt.plot(sizes, direct, "o-", label="NumPy arrays")
plt.plot(sizes, wrapped, "s--", label="Matrix wrapper")
plt.xlabel("Square matrix dimension n")
plt.ylabel("Median time (seconds)")
plt.title("Matrix multiplication timing (current environment)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("n | NumPy seconds | wrapper seconds")
for n, t1, t2 in results:
    print(f"{n:3d} | {t1:.6g} | {t2:.6g}")

## 5. Interpret the benchmark carefully

The wrapper delegates multiplication to NumPy, but it also performs input coercion, shape checks, and construction of a new Matrix result. For small arrays, Python-level overhead may dominate. For larger arrays, numerical computation becomes more important.

A fair report should record:
- Python and NumPy versions, platform and CPU details
- matrix dimensions, data type, and random seed
- warm-up procedure, repetitions, and summary statistic
- whether allocation and validation are included in the timed region
- accuracy checks and whether the same workload was used
- any limitations, noise, and repeated-run variation

Do not label the wrapper as faster than NumPy based on this experiment. The primary purpose is to measure overhead and learn reproducible benchmarking.

## Exercises

1. Change the random seed and confirm that timings vary.
2. Add a comparison for matrix addition.
3. Benchmark rectangular matrix multiplication.
4. Compare a loop-based multiplication implementation with NumPy for small matrices only; verify results before comparing speed.
5. Explain why benchmark results should not be committed as universal performance claims.
6. Run `python -m pytest --cov=matrixlib --cov-report=term-missing` and record the measured coverage separately from this notebook.